# 14 | Sampling Sensitivity and Minority-Class Coverage

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Research question:** How does the previous 500-windows-per-person cap affect retained class support compared with the full eligible HARTH development cohort? Audit caps of 250, 500 and 1,000 windows without fitting a classifier. No reserved Test participants are loaded. This is a design audit, not proof of model performance.

## 01. Set up the shared data and sampling audit functions

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the repository root and import the existing reusable code.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root == root.parent: raise RuntimeError("Run inside human-movement-intelligence.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_feature_cohort
from core.sampling_audit import audit_temporal_sampling
print("Sampling audit ready")

## 02. Recreate the complete 29-person eligible development cohort

The existing adapter independently windows each person's recording and removes out-of-task labels **after** time-safe segmentation. It also computes the existing 30 features; they are not used for model training in this notebook.

In [ ]:
# Keep the original reserved Test identities outside all data operations.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
features, metadata, overview, excluded = prepare_harth_feature_cohort(
    DEVELOPMENT_SUBJECTS, root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES, window_size=100, step_size=50, sampling_rate=50,
)
assert len(DEVELOPMENT_SUBJECTS) == 29
assert set(metadata["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
print("Full eligible development windows:", len(metadata))
print("Feature columns (not used for fitting):", features.shape[1])
display(overview[["subject_id", "eligible_windows", "excluded_windows"]])

## 03. Compare different label-blind caps with the full cohort

Sampling follows the **same evenly spaced eligible-window positions as Notebook 10**, independently for each participant. We do not select samples based on activity codes.

In [ ]:
# Audit source-window identities and class coverage without training.
CAPS = (250, 500, 1000)
selected, per_subject, class_counts = audit_temporal_sampling(
    metadata=metadata,
    caps=CAPS,
    classes=TASK_CLASSES,
    forbidden_subjects=RESERVED_TEST,
)
policies = ["full", "cap_250", "cap_500", "cap_1000"]
print("Window totals by policy:")
display(per_subject.groupby("policy")["windows"].sum().reindex(policies))
display(class_counts.pivot(index="label", columns="policy", values="windows")
        .reindex(index=TASK_CLASSES, columns=policies))

## 04. Quantify how many minority-class windows each cap retains

The table presents **retained fractions relative to full eligible development windows** for each activity. A larger global retained fraction does not guarantee that each subject or activity is adequately represented.

In [ ]:
# Build a class-by-policy retention table relative to the full cohort.
counts = class_counts.pivot(index="label", columns="policy", values="windows"
                           ).reindex(index=TASK_CLASSES, columns=policies)
retention = counts.div(counts["full"], axis=0).mul(100)
retention.index = [ACTIVITY_NAMES[k] for k in TASK_CLASSES]
display(retention.round(2))

# Use grouped bars for class retention across three sampling caps.
ax = retention[["cap_250", "cap_500", "cap_1000"]].plot(
    kind="bar", figsize=(12, 5), rot=35)
ax.set(title="HARTH class retention relative to full eligible development data",
       xlabel="Activity", ylabel="Retained windows (%)", ylim=(0, 105))
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 05. Audit the number of participants supporting each class

An activity may have many windows concentrated in only a few participants. Count **independent participant support**, not just windows.

In [ ]:
# Count participants contributing at least one window for each class/policy.
support = (
    selected.groupby(["policy", "label"])["subject_id"].nunique()
    .rename("subjects").reset_index()
    .pivot(index="label", columns="policy", values="subjects")
    .reindex(index=TASK_CLASSES, columns=policies).fillna(0).astype(int)
)
support.index = [ACTIVITY_NAMES[k] for k in TASK_CLASSES]
display(support)
ax = support.plot(kind="bar", figsize=(12, 5), rot=35)
ax.set(title="Independent participant support by sampling policy",
       xlabel="Activity", ylabel="Participants", ylim=(0, 29))
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 06. Inspect coverage specifically for low-frequency activities

Focus on Shuffling, Stairs ascending and Stairs descending. Identify people whose samples are lost entirely under each cap.

In [ ]:
# Compare per-subject retained counts for the three minority activities.
focus_codes = [3, 4, 5]
minority = selected.loc[selected["label"].isin(focus_codes)]
detail = pd.crosstab(
    index=[minority["subject_id"], minority["label"]],
    columns=minority["policy"],
).reindex(columns=policies, fill_value=0)
display(detail)
print("Source windows are sampled chronologically; no label balancing was applied.")

## 07. Quality gate and next modelling decision

This audit isolates the **data-selection policy**, but does not indicate how the different caps change validation Macro F1. That requires a later matched experimental comparison with an explicitly fixed subject protocol.

In [ ]:
# Confirm full source-row coverage, caps and protected test exclusions.
assert set(selected["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert not (set(selected["subject_id"]) & RESERVED_TEST)
assert len(selected.loc[selected["policy"] == "full"]) == len(metadata)
assert int(per_subject.loc[per_subject.policy == "cap_500", "windows"].sum()) == 14201
assert all((retention <= 100 + 1e-8).to_numpy().ravel())
assert class_counts.groupby("policy")["windows"].sum().loc["full"] == len(metadata)
print("Chronological sampling sensitivity audit: PASSED")
print("Reserved original Test loaded/scored: NO")
print("Models fitted in Stage 14: NONE")